In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd

sales = pd.read_pickle("../data/processed/nyc_sales_2003_2025_matched_v2.pkl")
num = sales.select_dtypes('number').columns
audit = pd.DataFrame({
    'dtype': sales.dtypes.astype(str),
    'null_%': (sales.isna().mean() * 100).round(1),
    'zero_%': (sales[num].eq(0).mean() * 100).round(1).reindex(sales.columns),
    'nunique': sales.nunique(),
})
print(audit.to_string())

for c in ['GROSS SQUARE FEET', 'LAND SQUARE FEET', 'YEAR BUILT', 'RESIDENTIAL UNITS']:
    if c in sales:
        ok = sales[c].notna() & (sales[c] > 0)
        print(f"\n{c}: share usable by year")
        print(ok.groupby(sales['SALE DATE'].dt.year).mean().round(2).to_string())

                                                    dtype  null_%  zero_%  nunique
BOROUGH                                             int64     0.0     0.0        5
NEIGHBORHOOD                                          str     0.0    <NA>      524
BUILDING CLASS CATEGORY                               str     0.0    <NA>      145
TAX CLASS AT PRESENT                                  str     0.2    <NA>       12
BLOCK                                             float64     0.0     0.0    13640
LOT                                               float64     0.0     0.0     5974
BUILDING CLASS AT PRESENT                             str     0.2    <NA>      220
ADDRESS                                               str     0.0    <NA>  1364778
APARTMENT NUMBER                                      str    30.6    <NA>    34716
ZIP CODE                                          float64     0.0     0.2      254
RESIDENTIAL UNITS                                 float64     6.4    23.3      532
COMM

In [2]:
cols = ['GROSS SQUARE FEET', 'LAND SQUARE FEET', 'YEAR BUILT', 'RESIDENTIAL UNITS']
usable = pd.DataFrame({c: (sales[c].notna() & (sales[c] > 0)) for c in cols})
usable['LAT/LON'] = sales['Latitude'].notna()
usable['ZIP'] = sales['ZIP CODE'].notna() & (sales['ZIP CODE'] > 0)
by_year = usable.groupby(sales['SALE DATE'].dt.year).mean().round(2)
by_year['n_sales'] = sales.groupby(sales['SALE DATE'].dt.year).size()
print(by_year.to_string())

           GROSS SQUARE FEET  LAND SQUARE FEET  YEAR BUILT  RESIDENTIAL UNITS  LAT/LON   ZIP  n_sales
SALE DATE                                                                                            
2003                    0.65              0.67        0.88               0.74     0.00  1.00   117008
2004                    0.63              0.65        0.89               0.71     0.00  1.00   124250
2005                    0.60              0.62        0.86               0.69     0.00  1.00   121675
2006                    0.58              0.60        0.86               0.71     0.00  1.00   114066
2007                    0.54              0.56        0.87               0.69     0.00  1.00   105296
2008                    0.51              0.53        0.87               0.69     0.00  1.00    89332
2009                    0.55              0.57        0.88               0.69     0.00  1.00    72803
2010                    0.58              0.61        0.85               0.77     

In [3]:
CC = ['04', '09', '10', '12', '13', '15', '17']
SEGMENTS = {**{c: 'HOUSE' for c in ['01', '02', '03']},
            **{c: 'CONDO_COOP' for c in CC},
            **{c: 'MULTIFAMILY' for c in ['07', '08', '14']}}
sales['SEGMENT'] = sales['BUILDING CLASS CATEGORY'].str.strip().str[:2].map(SEGMENTS)
sales['YEAR'] = sales['SALE DATE'].dt.year

d = sales[sales['SEGMENT'].notna() & (sales['SALE PRICE'] >= 10_000)].copy()
d['REL'] = d['SALE PRICE'] / d.groupby(['BOROUGH', 'SEGMENT', 'YEAR'])['SALE PRICE'].transform('median')
d = d[d['REL'] >= 0.25].copy()
for c in ['GROSS SQUARE FEET', 'LAND SQUARE FEET', 'YEAR BUILT']:
    d[c] = d[c].where(d[c] > 0)                       # 0 means unknown, not zero
d['y'] = np.log(d['SALE PRICE'])
print(f"modeling rows: {len(d):,} | distinct properties (MATCH_GROUP): {d['MATCH_GROUP'].nunique():,}")

# Property-grouped split: a MATCH_GROUP lands wholly in train, valid or test
rng = np.random.default_rng(42)
groups = d['MATCH_GROUP'].unique()
assign = pd.Series(rng.choice(['train', 'valid', 'test'], size=len(groups), p=[.70, .15, .15]), index=groups)
d['split'] = d['MATCH_GROUP'].map(assign)

print(d.groupby('split').size().to_string())
print("leak check, max splits per property:", d.groupby('MATCH_GROUP')['split'].nunique().max(), "(must be 1)")
print(d.groupby('split')[['y', 'GROSS SQUARE FEET', 'YEAR BUILT']].agg(['mean', lambda s: s.isna().mean()]).round(2).to_string())
print(d.groupby(['split', 'SEGMENT']).size().unstack().to_string())

modeling rows: 1,264,635 | distinct properties (MATCH_GROUP): 865,766
split
test     189348
train    885665
valid    189622
leak check, max splits per property: 1 (must be 1)
           y            GROSS SQUARE FEET            YEAR BUILT           
        mean <lambda_0>              mean <lambda_0>       mean <lambda_0>
split                                                                     
test   13.32        0.0           3619.87        0.5    1949.97       0.07
train  13.32        0.0           3803.06        0.5    1949.77       0.07
valid  13.32        0.0           3761.89        0.5    1949.84       0.07
SEGMENT  CONDO_COOP   HOUSE  MULTIFAMILY
split                                   
test          95908   84698         8742
train        449207  396142        40316
valid         96692   84308         8622


In [4]:
from src.hpi import repeat_sales_index, qlabels

pairs = pd.read_pickle("../data/processed/repeat_sales_pairs.pkl")
train_groups = set(assign.index[assign == 'train'])
ptr = pairs[pairs['MATCH_GROUP'].isin(train_groups)]
print(f"pairs from training properties: {len(ptr):,} of {len(pairs):,}")

idx = pd.DataFrame({b: repeat_sales_index(ptr[ptr['BOROUGH'] == b]) for b in [1, 2, 3, 4, 5]})
print("NaN quarters per borough:", idx.isna().sum().to_dict())
idx = idx.interpolate(limit_direction='both')
print(idx.loc[['2004Q4', '2010Q4', '2019Q4', '2025Q4']].round(1).to_string())

# time-adjusted target: log price minus log market level at the sale quarter
d['QLAB'] = d['SALE DATE'].dt.year.astype(str) + 'Q' + d['SALE DATE'].dt.quarter.astype(str)
lg = np.log(idx).stack()
d['LOG_HPI'] = lg.reindex(pd.MultiIndex.from_arrays([d['QLAB'], d['BOROUGH']])).to_numpy()
print("rows without an index value:", int(d['LOG_HPI'].isna().sum()))
d['y_adj'] = d['y'] - d['LOG_HPI']

def metrics(df, yhat):
    ape = np.abs(np.exp(yhat) - df['SALE PRICE'].to_numpy()) / df['SALE PRICE'].to_numpy()
    return pd.Series({'n': len(df), 'MdAPE': np.median(ape), 'PE10': (ape <= .10).mean(), 'PE20': (ape <= .20).mean()})

tr, va = d[d['split'] == 'train'], d[d['split'] == 'valid']

def lookup(table, df, cols):
    return table.reindex(pd.MultiIndex.from_frame(df[cols])).to_numpy()

def median_baseline(target):
    nb = lookup(tr.groupby(['NEIGHBORHOOD', 'SEGMENT'])[target].median(), va, ['NEIGHBORHOOD', 'SEGMENT'])
    bs = lookup(tr.groupby(['BOROUGH', 'SEGMENT'])[target].median(), va, ['BOROUGH', 'SEGMENT'])
    return np.where(np.isnan(nb), bs, nb)

b0 = median_baseline('y')                                  # no time adjustment
b1 = median_baseline('y_adj') + va['LOG_HPI'].to_numpy()   # with our index
res = pd.DataFrame({'0_no_time_adj': metrics(va, b0), '1_with_HPI': metrics(va, b1)}).T
print(res.round(3).to_string())
for seg in ['HOUSE', 'CONDO_COOP', 'MULTIFAMILY']:
    m_ = (va['SEGMENT'] == seg).to_numpy()
    print(seg, metrics(va[m_], b1[m_]).round(3).to_dict())

pairs from training properties: 255,067 of 364,338
NaN quarters per borough: {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
            1      2      3      4      5
2004Q4  116.5  114.3  112.9  110.1  112.4
2010Q4  143.6  111.8  117.5  106.1  111.3
2019Q4  193.4  193.1  225.9  191.1  160.9
2025Q4  196.5  249.8  276.7  231.1  215.9
rows without an index value: 0
                      n  MdAPE   PE10   PE20
0_no_time_adj  189622.0   0.29  0.189  0.364
1_with_HPI     189622.0   0.27  0.206  0.390
HOUSE {'n': 84308.0, 'MdAPE': 0.197, 'PE10': 0.274, 'PE20': 0.507}
CONDO_COOP {'n': 96692.0, 'MdAPE': 0.361, 'PE10': 0.15, 'PE20': 0.295}
MULTIFAMILY {'n': 8622.0, 'MdAPE': 0.359, 'PE10': 0.178, 'PE20': 0.321}


In [5]:
import time, xgboost as xgb

CATS = ['BOROUGH', 'NEIGHBORHOOD', 'BUILDING CLASS AT TIME OF SALE', 'TAX CLASS AT TIME OF SALE', 'SEGMENT', 'ZIP CODE']
NUMS = ['BLOCK', 'LOT', 'RESIDENTIAL UNITS', 'COMMERCIAL UNITS', 'TOTAL UNITS',
        'LAND SQUARE FEET', 'GROSS SQUARE FEET', 'YEAR BUILT']

X = d[CATS + NUMS].copy()
for c in CATS:
    X[c] = X[c].astype('category')
X['age_at_sale'] = d['YEAR'] - d['YEAR BUILT']
X['log_gross'] = np.log(d['GROSS SQUARE FEET'])
X['log_land'] = np.log(d['LAND SQUARE FEET'])
X['gross_per_unit'] = d['GROSS SQUARE FEET'] / d['TOTAL UNITS'].where(d['TOTAL UNITS'] > 0)

m_tr, m_va = (d['split'] == 'train').to_numpy(), (d['split'] == 'valid').to_numpy()
Xtr, ytr, Xva, yva = X[m_tr], d.loc[m_tr, 'y_adj'], X[m_va], d.loc[m_va, 'y_adj']

model = xgb.XGBRegressor(
    n_estimators=2000, learning_rate=0.08, max_depth=8, min_child_weight=10,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
    tree_method='hist', enable_categorical=True, n_jobs=-1,
    early_stopping_rounds=50, eval_metric='rmse', random_state=42)
t0 = time.time()
model.fit(Xtr, ytr, eval_set=[(Xtr.sample(100_000, random_state=0), ytr.loc[Xtr.sample(100_000, random_state=0).index]), (Xva, yva)], verbose=100)
print(f"trained in {(time.time() - t0) / 60:.1f} min | best iteration: {model.best_iteration}")

va_ = d[m_va]
pred_va = model.predict(Xva) + va_['LOG_HPI'].to_numpy()
tr_s = d[m_tr].sample(100_000, random_state=0)
pred_tr = model.predict(X.loc[tr_s.index]) + tr_s['LOG_HPI'].to_numpy()

out = pd.DataFrame({'train(100k sample)': metrics(tr_s, pred_tr), 'valid': metrics(va_, pred_va)}).T
print(out.round(3).to_string())
for seg in ['HOUSE', 'CONDO_COOP', 'MULTIFAMILY']:
    k = (va_['SEGMENT'] == seg).to_numpy()
    print(seg, metrics(va_[k], pred_va[k]).round(3).to_dict())

imp = pd.Series(model.get_booster().get_score(importance_type='gain')).sort_values(ascending=False)
print((imp / imp.sum()).head(15).round(3).to_string())
model.save_model("../data/processed/xgb_v1.json")

XGBoostError: [17:51:47] /Users/runner/work/xgboost/xgboost/src/data/columnar.h:144: Category index from DataFrame has floating point dtype, consider using strings or integers instead.
Stack trace:
  [bt] (0) 1   libxgboost.dylib                    0x0000000156ed96d0 dmlc::LogMessageFatal::~LogMessageFatal() + 124
  [bt] (1) 2   libxgboost.dylib                    0x0000000157045e10 auto xgboost::DispatchDType<void xgboost::DispatchDType<1, unsigned long xgboost::data::GetArrowNumericNames<std::__1::variant<enc::CatStrArrayView, xgboost::common::Span<unsigned char const, 18446744073709551615ul>, xgboost::common::Span<signed char const, 18446744073709551615ul>, xgboost::common::Span<unsigned short const, 18446744073709551615ul>, xgboost::common::Span<short const, 18446744073709551615ul>, xgboost::common::Span<unsigned int const, 18446744073709551615ul>, xgboost::common::Span<int const, 18446744073709551615ul>, xgboost::common::Span<unsigned long long const, 18446744073709551615ul>, xgbo

In [7]:
for c in CATS:
    s = d[c]
    if s.dtype == 'object' or str(s.dtype).startswith('str'):
        s = s.astype(object)                       # plain Python strings, not pandas' new str dtype
    X[c] = pd.Categorical(s)

print(X.dtypes.astype(str).value_counts().to_string())
print({c: str(X[c].cat.categories.dtype) for c in CATS})

small = xgb.XGBRegressor(n_estimators=5, tree_method='hist', enable_categorical=True)
small.fit(X[m_tr].iloc[:5000], d.loc[m_tr, 'y_adj'].iloc[:5000])
print("smoke test OK")

float64     12
category     6
{'BOROUGH': 'int64', 'NEIGHBORHOOD': 'object', 'BUILDING CLASS AT TIME OF SALE': 'object', 'TAX CLASS AT TIME OF SALE': 'float64', 'SEGMENT': 'object', 'ZIP CODE': 'float64'}


XGBoostError: [17:53:26] /Users/runner/work/xgboost/xgboost/src/data/columnar.h:144: Category index from DataFrame has floating point dtype, consider using strings or integers instead.
Stack trace:
  [bt] (0) 1   libxgboost.dylib                    0x0000000156ed96d0 dmlc::LogMessageFatal::~LogMessageFatal() + 124
  [bt] (1) 2   libxgboost.dylib                    0x0000000157045e10 auto xgboost::DispatchDType<void xgboost::DispatchDType<1, unsigned long xgboost::data::GetArrowNumericNames<std::__1::variant<enc::CatStrArrayView, xgboost::common::Span<unsigned char const, 18446744073709551615ul>, xgboost::common::Span<signed char const, 18446744073709551615ul>, xgboost::common::Span<unsigned short const, 18446744073709551615ul>, xgboost::common::Span<short const, 18446744073709551615ul>, xgboost::common::Span<unsigned int const, 18446744073709551615ul>, xgboost::common::Span<int const, 18446744073709551615ul>, xgboost::common::Span<unsigned long long const, 18446744073709551615ul>, xgbo

In [8]:
print("xgboost", xgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

FEATS = list(X.columns)
FTYPES = ['c' if c in CATS else 'q' for c in FEATS]

def to_np(df):
    out = np.empty((len(df), len(FEATS)), dtype=np.float32)
    for j, c in enumerate(FEATS):
        if c in CATS:
            codes = df[c].cat.codes.to_numpy().astype(np.float32)
            codes[codes < 0] = np.nan                       # -1 means missing
            out[:, j] = codes
        else:
            out[:, j] = df[c].to_numpy(dtype=np.float32, na_value=np.nan)
    return out

def make_model(n_estimators=2000):
    return xgb.XGBRegressor(
        n_estimators=n_estimators, learning_rate=0.08, max_depth=8, min_child_weight=10,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
        tree_method='hist', enable_categorical=True, feature_types=FTYPES, n_jobs=-1,
        early_stopping_rounds=50 if n_estimators > 10 else None, eval_metric='rmse', random_state=42)

# smoke test: tiny and fast
mini = make_model(5)
mini.fit(to_np(X[m_tr].iloc[:5000]), d.loc[m_tr, 'y_adj'].iloc[:5000])
print("smoke test OK")

# real fit
Xtr, ytr, Xva, yva = X[m_tr], d.loc[m_tr, 'y_adj'], X[m_va], d.loc[m_va, 'y_adj']
tr_s = d[m_tr].sample(100_000, random_state=0)
model = make_model()
t0 = time.time()
model.fit(to_np(Xtr), ytr, eval_set=[(to_np(X.loc[tr_s.index]), tr_s['y_adj']), (to_np(Xva), yva)], verbose=100)
print(f"trained in {(time.time() - t0) / 60:.1f} min | best iteration: {model.best_iteration}")

va_ = d[m_va]
pred_va = model.predict(to_np(Xva)) + va_['LOG_HPI'].to_numpy()
pred_tr = model.predict(to_np(X.loc[tr_s.index])) + tr_s['LOG_HPI'].to_numpy()

out = pd.DataFrame({'train(100k sample)': metrics(tr_s, pred_tr), 'valid': metrics(va_, pred_va)}).T
print(out.round(3).to_string())
for seg in ['HOUSE', 'CONDO_COOP', 'MULTIFAMILY']:
    k = (va_['SEGMENT'] == seg).to_numpy()
    print(seg, metrics(va_[k], pred_va[k]).round(3).to_dict())

booster = model.get_booster()
booster.feature_names = FEATS
imp = pd.Series(booster.get_score(importance_type='gain')).sort_values(ascending=False)
print((imp / imp.sum()).head(15).round(3).to_string())
model.save_model("../data/processed/xgb_v1.json")

xgboost 3.4.1 | pandas 3.0.5 | numpy 2.5.2
smoke test OK
[0]	validation_0-rmse:0.77018	validation_1-rmse:0.76985
[100]	validation_0-rmse:0.41597	validation_1-rmse:0.42689
[200]	validation_0-rmse:0.39718	validation_1-rmse:0.41332
[300]	validation_0-rmse:0.38602	validation_1-rmse:0.40599
[400]	validation_0-rmse:0.37857	validation_1-rmse:0.40166
[500]	validation_0-rmse:0.37260	validation_1-rmse:0.39838
[600]	validation_0-rmse:0.36741	validation_1-rmse:0.39567
[700]	validation_0-rmse:0.36315	validation_1-rmse:0.39368
[800]	validation_0-rmse:0.35957	validation_1-rmse:0.39216
[900]	validation_0-rmse:0.35647	validation_1-rmse:0.39097
[1000]	validation_0-rmse:0.35379	validation_1-rmse:0.39002
[1100]	validation_0-rmse:0.35134	validation_1-rmse:0.38920
[1200]	validation_0-rmse:0.34898	validation_1-rmse:0.38837
[1300]	validation_0-rmse:0.34691	validation_1-rmse:0.38780
[1400]	validation_0-rmse:0.34503	validation_1-rmse:0.38728
[1500]	validation_0-rmse:0.34329	validation_1-rmse:0.38684
[1600]	vali

In [9]:
print(out.round(3).to_string())
for seg in ['HOUSE', 'CONDO_COOP', 'MULTIFAMILY']:
    k = (va_['SEGMENT'] == seg).to_numpy()
    print(seg, metrics(va_[k], pred_va[k]).round(3).to_dict())
print("top features:", ", ".join(f"{k}={v:.3f}" for k, v in (imp / imp.sum()).head(10).items()))

                           n  MdAPE   PE10   PE20
train(100k sample)  100000.0  0.158  0.339  0.595
valid               189622.0  0.178  0.300  0.545
HOUSE {'n': 84308.0, 'MdAPE': 0.138, 'PE10': 0.375, 'PE20': 0.663}
CONDO_COOP {'n': 96692.0, 'MdAPE': 0.225, 'PE10': 0.241, 'PE20': 0.453}
MULTIFAMILY {'n': 8622.0, 'MdAPE': 0.238, 'PE10': 0.223, 'PE20': 0.434}
top features: SEGMENT=0.508, BOROUGH=0.086, ZIP CODE=0.062, GROSS SQUARE FEET=0.058, TOTAL UNITS=0.056, BUILDING CLASS AT TIME OF SALE=0.050, log_gross=0.044, TAX CLASS AT TIME OF SALE=0.032, NEIGHBORHOOD=0.023, RESIDENTIAL UNITS=0.021


In [10]:
# prior-sale features from the property's own history (strictly earlier sales only)
o = d.sort_values(['MATCH_GROUP', 'SALE DATE'])[['MATCH_GROUP', 'SALE DATE', 'y_adj']].copy()
g = o.groupby('MATCH_GROUP')
o['prev_y_adj'] = g['y_adj'].shift(1)
o['days_since_prev'] = (o['SALE DATE'] - g['SALE DATE'].shift(1)).dt.days
o['n_prior_sales'] = g.cumcount()
same_day = o['days_since_prev'] < 1
o.loc[same_day, ['prev_y_adj', 'days_since_prev']] = np.nan

for c in ['prev_y_adj', 'days_since_prev', 'n_prior_sales']:
    X[c] = o[c].reindex(X.index)
has_prior = X['prev_y_adj'].notna().to_numpy()
print(f"rows with a usable prior sale: {has_prior.mean():.1%}")
print(pd.Series(has_prior).groupby(d['split'].to_numpy()).mean().round(3).to_dict())

FEATS = list(X.columns)                         # to_np and make_model read these globals
FTYPES = ['c' if c in CATS else 'q' for c in FEATS]

Xtr, ytr, Xva, yva = X[m_tr], d.loc[m_tr, 'y_adj'], X[m_va], d.loc[m_va, 'y_adj']
tr_s = d[m_tr].sample(100_000, random_state=0)
model2 = make_model(3000)
t0 = time.time()
model2.fit(to_np(Xtr), ytr, eval_set=[(to_np(X.loc[tr_s.index]), tr_s['y_adj']), (to_np(Xva), yva)], verbose=500)
print(f"trained in {(time.time() - t0) / 60:.1f} min | best iteration: {model2.best_iteration}")

pred2 = model2.predict(to_np(Xva)) + va_['LOG_HPI'].to_numpy()
hp = has_prior[m_va]
rep = {'v1 all': metrics(va_, pred_va), 'v2 all': metrics(va_, pred2),
       'v1 has_prior': metrics(va_[hp], pred_va[hp]), 'v2 has_prior': metrics(va_[hp], pred2[hp]),
       'v1 no_prior': metrics(va_[~hp], pred_va[~hp]), 'v2 no_prior': metrics(va_[~hp], pred2[~hp])}
print(pd.DataFrame(rep).T.round(3).to_string())
for seg in ['HOUSE', 'CONDO_COOP', 'MULTIFAMILY']:
    k = (va_['SEGMENT'] == seg).to_numpy()
    print(seg, metrics(va_[k], pred2[k]).round(3).to_dict())
model2.save_model("../data/processed/xgb_v2.json")

rows with a usable prior sale: 31.1%
{'test': 0.311, 'train': 0.312, 'valid': 0.311}
[0]	validation_0-rmse:0.76906	validation_1-rmse:0.76867
[500]	validation_0-rmse:0.35147	validation_1-rmse:0.37717
[1000]	validation_0-rmse:0.33172	validation_1-rmse:0.36906
[1500]	validation_0-rmse:0.31940	validation_1-rmse:0.36548
[2000]	validation_0-rmse:0.31043	validation_1-rmse:0.36363
[2500]	validation_0-rmse:0.30310	validation_1-rmse:0.36258
[2999]	validation_0-rmse:0.29690	validation_1-rmse:0.36188
trained in 3.2 min | best iteration: 2993
                     n  MdAPE   PE10   PE20
v1 all        189622.0  0.178  0.300  0.545
v2 all        189622.0  0.159  0.338  0.591
v1 has_prior   58923.0  0.192  0.270  0.518
v2 has_prior   58923.0  0.130  0.402  0.674
v1 no_prior   130699.0  0.172  0.314  0.558
v2 no_prior   130699.0  0.174  0.309  0.554
HOUSE {'n': 84308.0, 'MdAPE': 0.131, 'PE10': 0.396, 'PE20': 0.679}
CONDO_COOP {'n': 96692.0, 'MdAPE': 0.185, 'PE10': 0.296, 'PE20': 0.528}
MULTIFAMILY {'n':

In [11]:
m_te = (d['split'] == 'test').to_numpy()
te_, Xte = d[m_te], X[m_te]
pred_te = model2.predict(to_np(Xte)) + te_['LOG_HPI'].to_numpy()
hp_te = has_prior[m_te]
rep = {'TEST all': metrics(te_, pred_te),
       'TEST has_prior': metrics(te_[hp_te], pred_te[hp_te]),
       'TEST no_prior': metrics(te_[~hp_te], pred_te[~hp_te])}
print(pd.DataFrame(rep).T.round(3).to_string())
for seg in ['HOUSE', 'CONDO_COOP', 'MULTIFAMILY']:
    k = (te_['SEGMENT'] == seg).to_numpy()
    print(seg, metrics(te_[k], pred_te[k]).round(3).to_dict())

price = te_['SALE PRICE'].to_numpy()
ratio = np.exp(pred_te) / price
print("median predicted/actual:", round(float(np.median(ratio)), 3))
ape = np.abs(ratio - 1)
by_year = pd.Series(ape).groupby(te_['YEAR'].to_numpy()).median().round(3)
print("MdAPE by sale year:", ", ".join(f"{y}:{v}" for y, v in by_year.items()))

# SHAP (exact TreeSHAP from XGBoost) on a 20k test sample
samp = te_.sample(20_000, random_state=1).index
dm = xgb.DMatrix(to_np(X.loc[samp]), feature_types=FTYPES, enable_categorical=True)
contrib = model2.get_booster().predict(dm, pred_contribs=True)
shap_abs = pd.Series(np.abs(contrib[:, :-1]).mean(axis=0), index=FEATS).sort_values(ascending=False)
print("mean |SHAP| (log-price units):", ", ".join(f"{k}={v:.3f}" for k, v in shap_abs.head(12).items()))

                       n  MdAPE   PE10   PE20
TEST all        189348.0  0.159  0.337  0.589
TEST has_prior   58849.0  0.131  0.397  0.670
TEST no_prior   130499.0  0.175  0.309  0.552
HOUSE {'n': 84698.0, 'MdAPE': 0.132, 'PE10': 0.393, 'PE20': 0.675}
CONDO_COOP {'n': 95908.0, 'MdAPE': 0.187, 'PE10': 0.295, 'PE20': 0.525}
MULTIFAMILY {'n': 8742.0, 'MdAPE': 0.229, 'PE10': 0.241, 'PE20': 0.445}
median predicted/actual: 0.982
MdAPE by sale year: 2003:0.18, 2004:0.161, 2005:0.148, 2006:0.148, 2007:0.154, 2008:0.161, 2009:0.171, 2010:0.176, 2011:0.175, 2012:0.167, 2013:0.167, 2014:0.168, 2015:0.176, 2016:0.163, 2017:0.158, 2018:0.136, 2019:0.152, 2020:0.147, 2021:0.155, 2022:0.152, 2023:0.155, 2024:0.159, 2025:0.159
mean |SHAP| (log-price units): ZIP CODE=0.247, BUILDING CLASS AT TIME OF SALE=0.217, NEIGHBORHOOD=0.127, GROSS SQUARE FEET=0.074, prev_y_adj=0.054, YEAR BUILT=0.053, age_at_sale=0.043, BLOCK=0.034, log_gross=0.034, LOT=0.034, BOROUGH=0.033, LAND SQUARE FEET=0.027


In [12]:
import os
os.makedirs("../data/processed/pg", exist_ok=True)
i64 = lambda s: pd.to_numeric(s, errors='coerce').astype('Int64')

sx = pd.DataFrame({'sale_id': d.index}, index=d.index)
sx['property_id']       = d['MATCH_GROUP']
sx['borough']           = d['BOROUGH']
sx['neighborhood']      = d['NEIGHBORHOOD']
sx['segment']           = d['SEGMENT']
sx['building_class']    = d['BUILDING CLASS AT TIME OF SALE'].str.strip()
sx['address']           = d['ADDRESS']
sx['unit']              = d['UNIT_FINAL']
sx['zip_code']          = i64(d['ZIP CODE'].where(d['ZIP CODE'] > 0))
sx['block']             = i64(d['BLOCK'])
sx['lot']               = i64(d['LOT'])
sx['bbl']               = i64(d['BBL'].where(d['BBL'] > 0))
sx['residential_units'] = i64(d['RESIDENTIAL UNITS'])
sx['total_units']       = i64(d['TOTAL UNITS'])
sx['land_sqft']         = d['LAND SQUARE FEET']
sx['gross_sqft']        = d['GROSS SQUARE FEET']
sx['year_built']        = i64(d['YEAR BUILT'])
sx['sale_date']         = d['SALE DATE'].dt.strftime('%Y-%m-%d')
sx['sale_price']        = d['SALE PRICE']
sx.to_csv("../data/processed/pg/sales.csv", index=False)

pairs = pd.read_pickle("../data/processed/repeat_sales_pairs.pkl")
px = pd.DataFrame({
    'property_id': pairs['MATCH_GROUP'], 'segment': pairs['SEGMENT'], 'borough': pairs['BOROUGH'],
    'neighborhood': pairs['NEIGHBORHOOD'], 'bbl': i64(pairs['BBL'].where(pairs['BBL'] > 0)),
    'date0': pairs['date0'].dt.strftime('%Y-%m-%d'), 'price0': pairs['price0'],
    'date1': pairs['date1'].dt.strftime('%Y-%m-%d'), 'price1': pairs['price1'],
    'days': pairs['days'], 'log_ratio': pairs['log_ratio']})
px.to_csv("../data/processed/pg/repeat_pairs.csv", index=False)

hp = pd.read_pickle("../data/processed/hpi_by_group_quarterly.pkl")
hl = hp.stack().rename('value').reset_index()
hl.columns = ['quarter', 'geography', 'value']
hl['quarter_start'] = [f"{q[:4]}-{3 * (int(q[5]) - 1) + 1:02d}-01" for q in hl['quarter']]
hl['geo_type'] = hl['geography'].map(lambda g: 'city' if g == 'City' else
                  ('segment' if g in ('HOUSE', 'CONDO_COOP', 'MULTIFAMILY') else 'borough'))
hl[['quarter', 'quarter_start', 'geography', 'geo_type', 'value']].to_csv("../data/processed/pg/hpi_quarterly.csv", index=False)
print(len(sx), len(px), len(hl))

1264635 364338 828


In [13]:
lab = d['UNIT_FINAL'].fillna('').str.match(r'^(RES|RCU|CONDO|CONDP|COOP|CO-OP|NY|HALL)')
unit_ok = ~((d['SEGMENT'] == 'CONDO_COOP') & (d['UNIT_FINAL'].isna() | lab))
ok_te = unit_ok.loc[te_.index].to_numpy()
print("share of test rows with a true unit-level history:", round(float((hp_te & ok_te).mean()), 3),
      "| building-level only:", round(float((hp_te & ~ok_te).mean()), 3))
for name, m in [('prior sale, same unit/property', hp_te & ok_te),
                ('prior sale, building-level only', hp_te & ~ok_te),
                ('no prior sale', ~hp_te)]:
    print(name, metrics(te_[m], pred_te[m]).round(3).to_dict())

share of test rows with a true unit-level history: 0.295 | building-level only: 0.016
prior sale, same unit/property {'n': 55872.0, 'MdAPE': 0.128, 'PE10': 0.406, 'PE20': 0.683}
prior sale, building-level only {'n': 2977.0, 'MdAPE': 0.256, 'PE10': 0.225, 'PE20': 0.417}
no prior sale {'n': 130499.0, 'MdAPE': 0.175, 'PE10': 0.309, 'PE20': 0.552}


In [14]:
from src.hpi import repeat_sales_index
pairs = pd.read_pickle("../data/processed/repeat_sales_pairs.pkl")
n = pairs.groupby('MATCH_GROUP')['MATCH_GROUP'].transform('size')
unit = pairs['MATCH_GROUP'].str.split('|').str[3]
lab = unit.fillna('').str.match(r'^(RES|RCU|COM|CONDO|CONDP|RSD|R-)')
bad = (n > 8) | (pairs['SEGMENT'].eq('CONDO_COOP') & lab)
print(f"pairs dropped: {int(bad.sum()):,} ({bad.mean():.2%})")

a, b = repeat_sales_index(pairs), repeat_sales_index(pairs[~bad])
cmp_ = pd.DataFrame({'all': a, 'clean': b})
cmp_['diff_pct'] = (cmp_['clean'] / cmp_['all'] - 1) * 100
print(cmp_.loc[['2004Q4', '2007Q4', '2010Q4', '2013Q4', '2016Q4', '2019Q4', '2022Q4', '2025Q4']].round(2).to_string())
print("largest difference across all 92 quarters (%):", round(cmp_['diff_pct'].abs().max(), 3))


pairs dropped: 731 (0.20%)
           all   clean  diff_pct
2004Q4  114.80  114.76     -0.04
2007Q4  149.41  149.36     -0.03
2010Q4  120.24  120.24     -0.01
2013Q4  147.21  147.20     -0.01
2016Q4  178.96  179.00      0.02
2019Q4  198.98  198.93     -0.02
2022Q4  216.85  216.85     -0.00
2025Q4  232.09  232.16      0.03
largest difference across all 92 quarters (%): 0.061
